# Reproducibility Notebook: Cluster-State Free Evolution vs SNAP

This notebook reproduces the saved results for the cluster-state-unitary decomposition study.
It compares a native gate set built from `Displacement`, `QubitRotation`, and `FreeEvolveCondPhase`
against an extended gate set that also includes `SNAP`.

Main questions:
1. Is free evolution alone enough to synthesize the target well?
2. Does SNAP reduce depth or entangling wait, or mainly add phase-correction freedom?
3. Which phases come from free evolution and which come from explicit phase gates?

The default path loads saved artifacts. Commented rerun cells are included for live recomputation.


## 1. Environment Setup

This cell finds the study root, imports the shared helpers, and exposes the saved-data directories.
The parameter cell below does not affect this discovery step.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

def find_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Study root not found.")

STUDY_ROOT = find_root(Path.cwd())
SCRIPTS_DIR = STUDY_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from common import (
    SNAP_S,
    SCREEN_MAXITER,
    SCREEN_SEED,
    TRUNCATION_LEVELS,
    ablation_sequences,
    base,
    build_family_sequence,
    evaluate_sequence_with_diagnostics,
    sequence_for_n_cav,
    sequence_from_payload,
    sequence_phase_budget,
)

DATA_DIR = STUDY_ROOT / "data"
FIG_DIR = STUDY_ROOT / "figures"
ARTIFACT_DIR = STUDY_ROOT / "artifacts"
load_json = lambda path: json.loads(Path(path).read_text(encoding="utf-8"))

np.set_printoptions(precision=4, suppress=True)
print(f"Study root: {STUDY_ROOT}")
print(f"Data dir:   {DATA_DIR}")
print(f"Figures:    {FIG_DIR}")
print(f"Artifacts:  {ARTIFACT_DIR}")


## 2. User-Tunable Parameters

All notebook knobs live here. If you change any value, rerun this cell and the next one.


In [ ]:
PARAMS = {
    "target_name": "cluster",
    "n_match": 1,
    "search_n_cav": 4,
    "validation_n_cav": list(TRUNCATION_LEVELS),
    "n_tr": int(base.N_TR),
    "native_family": "native_fe",
    "native_blocks": 6,
    "snap_family": "snap_interleaved",
    "snap_blocks": 6,
    "fe_ns": round(base.FE_DEFAULT_S * 1.0e9, 1),
    "snap_ns": round(SNAP_S * 1.0e9, 1),
    "seed": int(SCREEN_SEED),
    "maxiter": int(SCREEN_MAXITER),
    "thresholds": [0.90, 0.95, 0.97],
    "native_artifact": "best_native_sequence.json",
    "snap_artifact": "best_snap_sequence.json",
}

print(json.dumps(PARAMS, indent=2))


## 3. Derived Objects

This cell rebuilds the target, logical subspace, reference model, and template sequences from `PARAMS`.
Re-running Sections 2 -> 3 is enough to propagate parameter changes to the later rerun cells.


In [ ]:
target_matrix = np.asarray(
    base.make_target(PARAMS["target_name"], n_match=PARAMS["n_match"]),
    dtype=np.complex128,
)
logical_subspace = base.logical_subspace(PARAMS["search_n_cav"])
model = base.build_model(
    n_cav=max(PARAMS["validation_n_cav"]),
    n_tr=PARAMS["n_tr"],
)
native_template = build_family_sequence(
    family=PARAMS["native_family"],
    blocks=PARAMS["native_blocks"],
    n_cav=PARAMS["search_n_cav"],
    fe_duration_s=PARAMS["fe_ns"] * 1.0e-9,
    snap_duration_s=PARAMS["snap_ns"] * 1.0e-9,
)
snap_template = build_family_sequence(
    family=PARAMS["snap_family"],
    blocks=PARAMS["snap_blocks"],
    n_cav=PARAMS["search_n_cav"],
    fe_duration_s=PARAMS["fe_ns"] * 1.0e-9,
    snap_duration_s=PARAMS["snap_ns"] * 1.0e-9,
)

print("Logical basis:", base.LOGICAL_BASIS_ORDERING)
print("Target shape:", target_matrix.shape)
print("Reference truncation:", model.n_cav, model.n_tr)
print("Native template depth:", len(native_template.gates))
print("SNAP template depth:", len(snap_template.gates))


## 4. Saved Results and Threshold Comparison

This step loads the saved winners and the fidelity-threshold table.
It answers whether SNAP helps at practical fidelity targets.
Affected parameters: `thresholds`, `native_artifact`, `snap_artifact`.


In [ ]:
# --- Load saved results (default) ---
study = load_json(DATA_DIR / "study_summary.json")
thresholds = load_json(DATA_DIR / "threshold_summary.json")
best_native = study["best_native"]
best_snap = study["best_snap"]
snap_family = study["best_snap_family"]

print(f"Native winner: {best_native['case_id']}  F={best_native['fidelity']:.6f}  depth={best_native['gate_depth']}  wait={best_native['total_wait_time_ns']:.1f} ns")
print(f"SNAP winner:   {best_snap['case_id']}  F={best_snap['fidelity']:.6f}  depth={best_snap['gate_depth']}  wait={best_snap['total_wait_time_ns']:.1f} ns")

for thr in PARAMS["thresholds"]:
    key = f"{thr:.3f}"
    native_row = thresholds["native_fe"][key]
    snap_row = thresholds[snap_family][key]
    print()
    print(f"Threshold {thr:.2f}")
    print("  Native:", native_row["best_wait"] if native_row else "not reached")
    print("  SNAP:  ", snap_row["best_wait"] if snap_row else "not reached")
    if native_row and snap_row:
        dn = native_row["best_wait"]["gate_depth"] - snap_row["best_wait"]["gate_depth"]
        dt = native_row["best_wait"]["total_wait_time_ns"] - snap_row["best_wait"]["total_wait_time_ns"]
        print(f"  SNAP advantage: depth {dn:+d}, wait {dt:+.1f} ns")

assert abs(best_native["fidelity"] - 0.9639301025881614) < 1e-12
assert abs(best_snap["fidelity"] - 0.9880774560097466) < 1e-12
print()
print("Verification anchors matched the report.")


### Re-run With Current Parameters

These commented cells rerun one native and one FE+SNAP decomposition using the current parameter cell.


In [ ]:
# --- Re-run with current parameters ---
# native_seq = build_family_sequence(
#     family=PARAMS["native_family"],
#     blocks=PARAMS["native_blocks"],
#     n_cav=PARAMS["search_n_cav"],
#     fe_duration_s=PARAMS["fe_ns"] * 1.0e-9,
# )
# native_fit = base.fit_sequence(
#     native_seq,
#     n_cav=PARAMS["search_n_cav"],
#     seed=PARAMS["seed"],
#     init_guess="heuristic",
#     multistart=1,
#     maxiter=PARAMS["maxiter"],
# )
# native_eval = evaluate_sequence_with_diagnostics(native_fit["result"].sequence, n_cav=PARAMS["search_n_cav"])
# print("native fidelity:", native_eval["fidelity"])
#
# snap_seq = build_family_sequence(
#     family=PARAMS["snap_family"],
#     blocks=PARAMS["snap_blocks"],
#     n_cav=PARAMS["search_n_cav"],
#     fe_duration_s=PARAMS["fe_ns"] * 1.0e-9,
#     snap_duration_s=PARAMS["snap_ns"] * 1.0e-9,
# )
# snap_fit = base.fit_sequence(
#     snap_seq,
#     n_cav=PARAMS["search_n_cav"],
#     seed=PARAMS["seed"],
#     init_guess="heuristic",
#     multistart=1,
#     maxiter=PARAMS["maxiter"],
# )
# snap_eval = evaluate_sequence_with_diagnostics(snap_fit["result"].sequence, n_cav=PARAMS["search_n_cav"])
# print("snap fidelity:", snap_eval["fidelity"])


## 5. Phase Attribution

This step reconstructs the saved winners and separates native FE phase from explicit SNAP phase.
It is the direct answer to which parts of the target are implemented by waiting versus explicit phase gates.


In [ ]:
# --- Load saved results (default) ---
native_payload = load_json(ARTIFACT_DIR / PARAMS["native_artifact"])
snap_payload = load_json(ARTIFACT_DIR / PARAMS["snap_artifact"])

native_seq = sequence_from_payload(native_payload["sequence_payload"], n_cav=PARAMS["search_n_cav"])
snap_seq = sequence_from_payload(snap_payload["sequence_payload"], n_cav=PARAMS["search_n_cav"])
native_budget = sequence_phase_budget(native_seq)
snap_budget = sequence_phase_budget(snap_seq)

def show_budget(name, budget):
    print(name)
    for row in budget["fe_rows"]:
        wait_ns = float(row.get("wait_time", row.get("duration", 0.0))) * 1.0e9
        dphi_pi = float(row["delta_phi"][1]) / np.pi
        print(f"  {row['gate_name']}: wait={wait_ns:6.1f} ns  conditional phase={dphi_pi:6.3f} pi")
    if budget["snap_rows"]:
        for row in budget["snap_rows"]:
            rel = [round(float(v) / np.pi, 3) for v in row["relative_phases_rad"]]
            print(f"  {row['gate_name']}: SNAP relative phases/pi = {rel}")
    else:
        print("  No SNAP layers.")
    print(f"  Total FE phase/pi:   {budget['total_fe_logical_delta_phi_rad'] / np.pi:6.3f}")
    print(f"  Total SNAP phase/pi: {budget['total_snap_logical_relative_phase_rad'] / np.pi:6.3f}")
    print()

show_budget("Best native", native_budget)
show_budget("Best FE+SNAP", snap_budget)


### Re-run Phase Attribution

This rerun path recomputes the FE+SNAP family and prints a fresh phase table for the current parameters.


In [ ]:
# --- Re-run with current parameters ---
# seq = build_family_sequence(
#     family=PARAMS["snap_family"],
#     blocks=PARAMS["snap_blocks"],
#     n_cav=PARAMS["search_n_cav"],
#     fe_duration_s=PARAMS["fe_ns"] * 1.0e-9,
#     snap_duration_s=PARAMS["snap_ns"] * 1.0e-9,
# )
# fit = base.fit_sequence(
#     seq,
#     n_cav=PARAMS["search_n_cav"],
#     seed=PARAMS["seed"],
#     init_guess="heuristic",
#     multistart=1,
#     maxiter=PARAMS["maxiter"],
# )
# for row in fit["result"].sequence.phase_decomposition(n_match=PARAMS["n_match"]):
#     print(row)


## 6. Validation

This step reloads the ablations, truncation replay, and report figures.
Affected parameters: `validation_n_cav`, `native_artifact`, and `snap_artifact`.


In [ ]:
# --- Load saved results (default) ---
ablations = load_json(DATA_DIR / "ablation_results.json")
validation = load_json(DATA_DIR / "validation_summary.json")

print("Ablations")
for family, variants in ablations.items():
    print()
    print(family)
    for label, row in variants.items():
        print(f"  {label:<22} F={row['fidelity']:.6f}  depth={row['gate_depth']:2d}  wait={row['total_wait_time_ns']:.1f} ns")

print()
print("Truncation replay")
for family, rows in validation.items():
    print()
    print(family)
    for row in rows:
        print(f"  n_cav={row['n_cav']}: F={row['fidelity']:.6f}  leak_avg={row['leakage_average']:.6f}")

assert ablations["best_native"]["without_free_evolution"]["fidelity"] < 0.2
assert ablations["best_snap"]["without_free_evolution"]["fidelity"] < 0.4

fig, ax = plt.subplots(figsize=(6.4, 4.0))
for family, rows in validation.items():
    ax.plot([r["n_cav"] for r in rows], [r["fidelity"] for r in rows], marker="o", label=family)
ax.set_xlabel("Cavity truncation")
ax.set_ylabel("Logical fidelity")
ax.set_title("Saved truncation replay")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


### Re-run Validation

This rerun path replays the selected sequence across the current truncation list and rebuilds ablation variants.


In [ ]:
# --- Re-run with current parameters ---
# seq = sequence_from_payload(snap_payload["sequence_payload"], n_cav=PARAMS["search_n_cav"])
# for n_cav in PARAMS["validation_n_cav"]:
#     row = evaluate_sequence_with_diagnostics(sequence_for_n_cav(seq, n_cav=n_cav), n_cav=n_cav)
#     print("snap", n_cav, row["fidelity"])
#
# for label, variant in ablation_sequences(seq).items():
#     row = evaluate_sequence_with_diagnostics(variant, n_cav=PARAMS["search_n_cav"])
#     print(label, row["fidelity"])


## 7. Key Figures

These are the report figures regenerated by the study scripts, including the larger-truncation improvement-pass figures.


In [ ]:
# --- Load saved results (default) ---
for name in [
    "fidelity_vs_depth.png",
    "wait_vs_fidelity.png",
    "sequence_ablations.png",
    "phase_budget_comparison.png",
    "truncation_validation.png",
    "improvement_reoptimization_comparison.png",
    "improvement_truncation_noise_summary.png",
]:
    print()
    print(f"--- {name} ---")
    display(Image(filename=str(FIG_DIR / name), width=720))


### Re-run a Quick Comparison Figure

After uncommenting the live rerun cell above, this cell can be used to draw a simple native-vs-SNAP bar chart from the fresh results.


In [ ]:
# --- Re-run with current parameters ---
# fig, ax = plt.subplots(figsize=(5.5, 4.0))
# ax.bar(["native", "snap"], [native_eval["fidelity"], snap_eval["fidelity"]], color=["#4477AA", "#228833"])
# ax.set_ylabel("Logical fidelity")
# ax.set_title("Live rerun comparison")
# ax.set_ylim(0.0, 1.0)
# plt.show()


## 8. Summary

This notebook reproduced the saved winners, the threshold comparison, the phase split between free evolution and SNAP, the ablations, the truncation replay, the larger-truncation improvement pass, and the report figures.
The same qualitative conclusions as the paper follow: FE is the indispensable entangler, tail-only SNAP is ineffective, interleaved SNAP helps at the `0.95` threshold, and the best FE+SNAP solution gets higher fidelity by paying for more explicit phase-control complexity. The larger-truncation extension strengthens that ranking rather than overturning it.

Caveats:
- The original headline fidelities are ideal-gate results from the `n_cav=4` family sweep.
- A targeted improvement pass re-optimizes only the two depth-6 winners at `n_cav=6`; the full family sweep has not yet been repeated at larger truncation.
- Coherent replay now exists through the `cqed_sim` pulse-unitary backend, but this is still a control surrogate rather than a compiled waveform-bridge replay.
- The noisy validation is a documented Lindblad surrogate built from the per-gate pulse unitaries because direct `NoiseSpec` replay is not available for FE/SNAP sequences on this build.

| Tunable parameter | Default | Effect |
| --- | --- | --- |
| `target_name` | `cluster` | Selects the logical target unitary. |
| `n_match` | `1` | Chooses the cluster-target convention used by `make_target`. |
| `search_n_cav` | `4` | Sets the optimization truncation for rerun fits. |
| `validation_n_cav` | `[4, 6, 8]` | Sets the truncations used in the replay check. |
| `n_tr` | `2` | Sets the transmon truncation in the derived model. |
| `native_blocks` | `6` | Sets the no-SNAP ansatz depth in reruns. |
| `snap_blocks` | `6` | Sets the FE+SNAP ansatz depth in reruns. |
| `fe_ns` | `176.1` | Sets the nominal FE wait prior for reruns. |
| `snap_ns` | `200.0` | Sets the nominal SNAP duration for reruns. |
| `seed` | `17` | Changes the local-optimization initialization. |
| `maxiter` | `20` | Changes the speed-vs-quality tradeoff in reruns. |
| `thresholds` | `[0.9, 0.95, 0.97]` | Sets the checkpoints used to compare native FE and FE+SNAP. |
| `native_artifact` | `best_native_sequence.json` | Selects which saved no-SNAP winner is loaded by default. |
| `snap_artifact` | `best_snap_sequence.json` | Selects which saved FE+SNAP winner is loaded by default. |
